# Lab 3: Text Classification with RNN

In this lab, we build a complete end-to-end text classification pipeline. The key difference from previous labs is that the `TextVectorization` layer is **embedded inside the model**, so the final model accepts raw text strings directly -- no external preprocessing required.

## Objectives

- Build an end-to-end model with TextVectorization inside the Keras model
- Apply Bidirectional LSTM with Dropout regularization
- Use EarlyStopping to prevent overfitting
- Plot and interpret learning curves
- Evaluate with sample predictions and confidence scores
- Save the model for reuse in Lab 4

In [ ]:
# Run this cell in Google Colab to install dependencies
# Skip if running locally with uv
import sys
if 'google.colab' in sys.modules:
    !pip install -q keras torch torchvision gradio python-dotenv datasets transformers huggingface_hub
    print('Dependencies installed!')

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import keras
import numpy as np
import matplotlib.pyplot as plt

print(f"Keras version: {keras.__version__}")
print(f"Keras backend: {keras.backend.backend()}")

## 1. Load and Prepare the IMDB Dataset

In [ ]:
# Load IMDB dataset and decode to text
(x_train_enc, y_train), (x_test_enc, y_test) = keras.datasets.imdb.load_data()

word_index = keras.datasets.imdb.get_word_index()
reverse_word_index = {value + 3: key for key, value in word_index.items()}
reverse_word_index[0] = "<pad>"
reverse_word_index[1] = "<start>"
reverse_word_index[2] = "<unk>"
reverse_word_index[3] = "<unused>"

def decode_review(encoded_review):
    return " ".join(reverse_word_index.get(i, "?") for i in encoded_review)

x_train_text = np.array([decode_review(seq) for seq in x_train_enc])
x_test_text = np.array([decode_review(seq) for seq in x_test_enc])

print(f"Training samples: {len(x_train_text)}")
print(f"Test samples: {len(x_test_text)}")
print(f"\nSample review (first 200 chars): {x_train_text[0][:200]}")

## 2. Build the End-to-End Model

The model includes `TextVectorization` as its first layer. This means:
- The model accepts raw text strings as input
- No external preprocessing is needed at inference time
- The saved model is fully self-contained

### Architecture

```
Input (raw text string)
    -> TextVectorization(max_tokens=10000, output_sequence_length=200)
    -> Embedding(10000, 128)
    -> Bidirectional(LSTM(64))
    -> Dropout(0.5)
    -> Dense(1, sigmoid)
```

In [ ]:
# Hyperparameters
MAX_TOKENS = 10000
MAX_LENGTH = 200
EMBEDDING_DIM = 128

# Create and adapt the TextVectorization layer
text_vectorizer = keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_sequence_length=MAX_LENGTH,
    output_mode="int",
)
text_vectorizer.adapt(x_train_text)

print(f"Vocabulary size: {len(text_vectorizer.get_vocabulary())}")

In [ ]:
# Build the end-to-end model with TextVectorization inside
model = keras.Sequential([
    keras.layers.Input(shape=(1,), dtype="string"),
    text_vectorizer,
    keras.layers.Embedding(input_dim=MAX_TOKENS, output_dim=EMBEDDING_DIM),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

model.summary()

## 3. Train with EarlyStopping

`EarlyStopping` monitors `val_loss` and stops training when it has not improved for `patience=3` consecutive epochs. The `restore_best_weights=True` flag ensures we keep the best model.

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1,
)

history = model.fit(
    x_train_text,
    y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1,
)

## 4. Plot Learning Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history["loss"], label="Train Loss", marker="o")
axes[0].plot(history.history["val_loss"], label="Val Loss", marker="s")
axes[0].set_title("Loss Over Epochs")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history.history["accuracy"], label="Train Accuracy", marker="o")
axes[1].plot(history.history["val_accuracy"], label="Val Accuracy", marker="s")
axes[1].set_title("Accuracy Over Epochs")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Identify the best epoch
best_epoch = np.argmin(history.history["val_loss"]) + 1
best_val_acc = history.history["val_accuracy"][best_epoch - 1]
print(f"\nBest epoch: {best_epoch}")
print(f"Best validation accuracy: {best_val_acc:.4f}")

## 5. Evaluate the Model

In [ ]:
# Evaluate on the full test set
test_loss, test_acc = model.evaluate(x_test_text, y_test, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

In [ ]:
# Sample predictions with confidence scores
sample_indices = np.random.choice(len(x_test_text), 10, replace=False)
sample_texts = x_test_text[sample_indices]
sample_labels = y_test[sample_indices]
sample_preds = model.predict(sample_texts, verbose=0).flatten()

print(f"{'Pred':>8} {'True':>6} {'Conf':>8}  Review (first 80 chars)")
print("-" * 120)
for text, label, pred in zip(sample_texts, sample_labels, sample_preds):
    pred_label = "Pos" if pred > 0.5 else "Neg"
    true_label = "Pos" if label == 1 else "Neg"
    confidence = pred if pred > 0.5 else 1 - pred
    correct = "ok" if (pred > 0.5) == (label == 1) else "WRONG"
    print(f"{pred_label:>8} {true_label:>6} {confidence:>7.1%}  {text[:80]}  [{correct}]")

In [ ]:
# Interactive custom review testing
custom_reviews = [
    "This is the best movie I have ever seen in my entire life!",
    "Absolutely horrible. The plot made no sense and the acting was wooden.",
    "It was an okay film. Some parts were good, others not so much.",
    "A stunning visual experience with a deeply moving story. Highly recommended.",
    "I fell asleep halfway through. Boring and predictable.",
]

preds = model.predict(np.array(custom_reviews), verbose=0).flatten()

print("Custom Review Predictions")
print("=" * 80)
for review, pred in zip(custom_reviews, preds):
    sentiment = "POSITIVE" if pred > 0.5 else "NEGATIVE"
    confidence = pred if pred > 0.5 else 1 - pred
    bar = "#" * int(confidence * 40)
    print(f"\n{sentiment} ({confidence:.1%}) {bar}")
    print(f"  {review}")

## 6. Save Model for Lab 4

We save the trained model so it can be loaded and used as a tool in the AI Agent Chatbot (Lab 4).

In [ ]:
# Save the model
model_path = "imdb_sentiment_model.keras"
model.save(model_path)
print(f"Model saved to: {model_path}")

# Verify the saved model can be loaded and produces the same results
loaded_model = keras.saving.load_model(model_path)
test_text = np.array(["This movie was great!"])
original_pred = model.predict(test_text, verbose=0)
loaded_pred = loaded_model.predict(test_text, verbose=0)
print(f"Original prediction: {original_pred[0][0]:.4f}")
print(f"Loaded prediction:   {loaded_pred[0][0]:.4f}")
print(f"Match: {np.allclose(original_pred, loaded_pred, atol=1e-5)}")

## 7. Gradio Interface

Launch an interactive Gradio interface to test the sentiment classifier. Enter any text and see the sentiment prediction with a confidence bar.

In [ ]:
import gradio as gr

def predict_sentiment(text):
    """Predict sentiment for a given text input with confidence."""
    if not text.strip():
        return {"Positive": 0.5, "Negative": 0.5}
    pred = model.predict(np.array([text]), verbose=0)[0][0]
    return {"Positive": float(pred), "Negative": float(1 - pred)}

demo = gr.Interface(
    fn=predict_sentiment,
    inputs=gr.Textbox(
        lines=5,
        placeholder="Type a movie review here...",
        label="Movie Review",
    ),
    outputs=gr.Label(num_top_classes=2, label="Sentiment Prediction"),
    title="Text Classification with Bidirectional LSTM",
    description=(
        "This model uses an end-to-end pipeline: TextVectorization -> Embedding -> "
        "Bidirectional LSTM -> Dropout -> Dense. Enter a movie review to see the "
        "sentiment prediction with confidence."
    ),
    examples=[
        ["An absolutely phenomenal film with stellar performances from the entire cast."],
        ["This was the worst movie I have ever had the misfortune of watching."],
        ["The movie was alright. Nothing groundbreaking but it kept me entertained."],
        ["Beautifully shot and emotionally resonant. A true work of art."],
        ["Predictable plot, bad dialogue, and terrible special effects."],
    ],
)

demo.launch()